In [1]:
using CSV, DataFrames,JSON, CodecZlib

In [2]:

#sequence,locus,stop_codon,vj_in_frame,v_frameshift,productive,rev_comp,complete_vdj,v_call,d_call,j_call,sequence_alignment,germline_alignment,sequence_alignment_aa,germline_alignment_aa,v_alignment_start,v_alignment_end,d_alignment_start,d_alignment_end,j_alignment_start,j_alignment_end,v_sequence_alignment,v_sequence_alignment_aa,v_germline_alignment,v_germline_alignment_aa,d_sequence_alignment,d_sequence_alignment_aa,d_germline_alignment,d_germline_alignment_aa,j_sequence_alignment,j_sequence_alignment_aa,j_germline_alignment,j_germline_alignment_aa,fwr1,fwr1_aa,cdr1,cdr1_aa,fwr2,fwr2_aa,cdr2,cdr2_aa,fwr3,fwr3_aa,fwr4,fwr4_aa,cdr3,cdr3_aa,junction,junction_length,junction_aa,junction_aa_length,v_score,d_score,j_score,v_cigar,d_cigar,j_cigar,v_support,d_support,j_support,v_identity,d_identity,j_identity,v_sequence_start,v_sequence_end,v_germline_start,v_germline_end,d_sequence_start,d_sequence_end,d_germline_start,d_germline_end,j_sequence_start,j_sequence_end,j_germline_start,j_germline_end,fwr1_start,fwr1_end,cdr1_start,cdr1_end,fwr2_start,fwr2_end,cdr2_start,cdr2_end,fwr3_start,fwr3_end,fwr4_start,fwr4_end,cdr3_start,cdr3_end,np1,np1_length,np2,np2_length,c_region,Redundancy,ANARCI_numbering,ANARCI_status
function process_data(path::String)
    
    df = CSV.File(path, 
              skipto=3,  # Skip the metadata line
              header=2,  # Use the second line as header
              delim=',', 
              missingstring=["", "NA"],
              ignoreemptyrows=true,
              silencewarnings=true) |> DataFrame;;
   return df
end

process_data (generic function with 1 method)

In [3]:

function filter_data(df::DataFrame,src::String, dst::String,  filter::Matrix{Any}, target_length::Int64)
    denied = Int64[] #rows that we dont use
  
    println("file: ", src)
#=
    for (index, seq) in enumerate(df.sequence_alignment_aa)
        if length(seq) != target_length && !(index in denied)
            push!(denied, index)
        end
    end
    println(length(denied),"/",nrow(df)," removed with condition: Sequence_alignment_aa = ", target_length) 

    if length(denied) == nrow(df)
        return nrow(df)-length(denied) 
    end
    =#
    fixed_length_df = df[setdiff(1:nrow(df), denied), :]

    for i in 1:size(filter, 2)
        header = filter[1,i]
        undesired_value = filter[2,i]
        
        if header in names(df)
            for (index, value) in enumerate(df[!, header])
                if value == undesired_value && !(index in denied)
                    push!(denied, index)
                end
            end
            println(length(denied),"/",nrow(df)," removed with condition: " ,filter[1,i], " = ", filter[2,i])  
        
            if length(denied) == nrow(df)
                break
            end
        end     
    end

    
    

    if length(denied) != nrow(df)
        filtered_df = df[Not(denied), :]
        # Extract only the "sequence_alignment_aa" column
        sequence_column = filtered_df[!, "sequence_alignment_aa"]

        path= dst[1:end-6] * "txt"
        # Write only this column to the file
        open(path, "w") do io
            for sequence in sequence_column
                println(io, sequence)
            end
        end 
    else
        println("all rows denied, not copying file over")
    end

    println("")
    
    return nrow(df)-length(denied)   
end

function filter_data(directory::String, filtered_dir::String, filter::Matrix{Any}, target_length::Int64)
    unfiltered_data = readdir(directory)
    
    accepted = 0
    total_rows = 0
    
    i = 300
    
    for i in eachindex(unfiltered_data)
        file = unfiltered_data[i]
        if file == ".ipynb_checkpoints" || file == "bulk_download (5).sh"
            continue
        end
        path = "antibody_data/unfiltered_data/$file"
        
        
        df = process_data(path)
        
        total_rows += filter_data(df, path, "$filtered_dir$file",filter, target_length)
        println( i, " / ", length(unfiltered_data))
    end
    println("total amount of rows of data accepted: ", total_rows)
end


filter_data (generic function with 2 methods)

In [ ]:
directory = "antibody_data/unfiltered_data/"
filtered_dir = "antibody_data/filtered_data/"
filter_parameters = ["productive" "complete_vdj" ; false false] #2D array of headers we want to filter and what value to filter away
target_length = 122

filter_data(directory, filtered_dir, filter_parameters, target_length)

file: antibody_data/unfiltered_data/ERR3004229_1_Heavy_Bulk.csv.gz
0/1094 removed with condition: productive = false
782/1094 removed with condition: complete_vdj = false

1 / 487
file: antibody_data/unfiltered_data/ERR3004229_1_Heavy_IGHA.csv.gz
0/7 removed with condition: productive = false
2/7 removed with condition: complete_vdj = false

2 / 487
file: antibody_data/unfiltered_data/ERR3004229_1_Heavy_IGHG.csv.gz
0/13 removed with condition: productive = false
1/13 removed with condition: complete_vdj = false

3 / 487
file: antibody_data/unfiltered_data/ERR3004229_1_Heavy_IGHM.csv.gz
0/265055 removed with condition: productive = false
21497/265055 removed with condition: complete_vdj = false

4 / 487
file: antibody_data/unfiltered_data/ERR3004230_1_Heavy_Bulk.csv.gz
0/864 removed with condition: productive = false
412/864 removed with condition: complete_vdj = false

5 / 487
file: antibody_data/unfiltered_data/ERR3004230_1_Heavy_IGHA.csv.gz
0/25412 removed with condition: productive 

In [1]:
filtered_dir = "antibody_data/filtered_data/"
filtered_data = readdir(filtered_dir)[2:end]
total_rows = 0
filtered_data = filtered_dir .* filtered_data
for i in eachindex(filtered_data)
 total_rows += countlines(filtered_data[i])
end

println(total_rows)

2560363
